# 03 – Explorative Datenanalyse (EDA)

**CRISP-DM-Phase:** Data Understanding (vertieft) / Vorbereitung der Modellierung.

**Methodischer Vorbehalt:** EDA ist **hypothesengenerierend**, nicht
-bestätigend. Alle hier sichtbaren Zusammenhänge sind **korrelativ**, nicht kausal.

Zu **jeder** Grafik beantworten wir: *Was zeigt sie? Warum ist sie relevant? Welche
Interpretation ist möglich? Welche Grenzen hat sie?*

In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from src import config, utils, eda

feat = utils.load_processed("feature_matrix")
print("Feature-Matrix:", feat.shape)

Feature-Matrix: (307511, 161)


## 1. Verteilung der Zielvariable

* **Was es zeigt:** Das Diagramm visualisiert die absolute und relative Häufigkeit unserer Zielvariable (`TARGET`). Es vergleicht die Fälle **ohne Kreditausfall** (0: 282.686 Fälle / 91,9 %) mit den Fällen **mit Kreditausfall** (1: 24.825 Fälle / 8,1 %).
* **Warum das wichtig ist:** Die Kenntnis dieser Verteilung ist ein essenzieller erster Schritt, da sie die Wahl der Bewertungsmetriken (Evaluation Metrics) und die Strategie für das Modelltraining (z. B. den Umgang mit Klassenungleichgewichten) maßgeblich bestimmt.
* **Interpretation & Konsequenzen:** Es liegt ein starkes **Klassenungleichgewicht (Imbalanced Data)** vor. Da nur etwa 8 % der Fälle positiv (Ausfall) sind, ist die klassische Metrik *Accuracy* (Gesaugtgenauigkeit) hier irreführend. Ein Modell, das blind immer "0" vorhersagt, hätte bereits eine scheinbar exzellente Genauigkeit von 91,9 %, wäre in der Praxis aber absolut nutzlos. Daher müssen alternative Metriken wie *Precision, Recall, F1-Score* oder *ROC-AUC* herangezogen werden. Beim Modelltraining sollten zudem Methoden wie Klassengewichtung (Class Weights) oder Resampling (Oversampling/Undersampling) erwogen werden.
* **Grenzen der Darstellung:** Die Grafik zeigt lediglich die eindimensionale Randverteilung der Zielvariable. Sie liefert noch keine Erkenntnisse darüber, welche Merkmale (Features) mit einem Kreditausfall korrelieren oder diesen tatsächlich verursachen.

In [2]:
fig = eda.plot_target_distribution(feat); plt.show()

21:14:29 | INFO    | Abbildung gespeichert: C:\Users\annis\OneDrive - Hochschule Heilbronn\MWI\Sem1\ML\files\reports\figures\eda\target_distribution.png
C:\Users\annis\AppData\Local\Temp\ipykernel_22300\4060539775.py:1: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig = eda.plot_target_distribution(feat); plt.show()


## 2. Analyse der fehlenden Werte (Missing Values)

* **Was es zeigt:** Das Balkendiagramm visualisiert die Top 20 Merkmale (Features) mit dem höchsten prozentualen Anteil an fehlenden Werten. Die Spitzenreiter weisen extrem hohe Lücken von fast 70 % auf.
* **Warum das wichtig ist:** Die systematische Untersuchung fehlender Werte ist ein kritischer Schritt im Data Preprocessing. Das Ausmaß der Lücken steuert maßgeblich die Feature-Auswahl und entscheidet darüber, ob Imputationsstrategien (Auffüllen von Daten) angewandt werden oder ob Variablen komplett verworfen werden müssen.
* **Interpretation & Konsequenzen:** * **Art der fehlenden Daten:** Die Grafik zeigt, dass in diesem Datensatz vor allem detaillierte Informationen zu den Wohn- und Gebäudeverhältnissen (Variablen mit Endungen wie `_AVG`, `_MODE`, `_MEDI` z. B. bei `COMMONAREA` oder `LIVINGAPARTMENTS`) sowie das Alter des Autos (`OWN_CAR_AGE`) massiv fehlen (zwischen 60 % und 70 %).
  * **Herausforderung bei der Imputation:** Bei einem Fehlwertanteil von fast 70 % ist eine klassische Imputation (z. B. das Auffüllen mit dem Mittelwert) stark fehleranfällig und verzerrt die Datenverteilung. Solche Spalten werden in der Praxis häufig aus dem Modellbau ausgeschlossen.
  * **Informationsgehalt des Fehlens:** Das Fehlen von Daten ist jedoch nicht immer nur ein "Fehler", sondern kann selbst eine wertvolle Information (ein Prädiktor) sein. So könnte die Weigerung oder Unfähigkeit, Gebäudedaten anzugeben, stark mit dem Kreditausfallrisiko korrelieren. Dies lässt sich als Feature nutzen, indem man z. B. binäre Flags erstellt (`has_building_info`) oder zählt, wie viele Werte pro Kunde insgesamt fehlen (`missing_values_count`).

In [3]:
fig = eda.plot_missingness(feat, top=20); plt.show()

21:14:48 | INFO    | Abbildung gespeichert: C:\Users\annis\OneDrive - Hochschule Heilbronn\MWI\Sem1\ML\files\reports\figures\eda\missingness.png
C:\Users\annis\AppData\Local\Temp\ipykernel_22300\3915583392.py:1: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig = eda.plot_missingness(feat, top=20); plt.show()


## 3. Verteilungen zentraler numerischer Features

**Was:** Histogramme von Einkommen, Kredit, Alter, Ratios, externem Score, Verzugsquote.  
**Warum:** Schiefe/Ausreißer erkennen.  

## Interpretation der Verteilungen zentraler numerischer Features

Die vorliegenden Histogramme geben Aufschluss über die Verteilung wichtiger Merkmale im Datensatz. Sie helfen dabei, Ausreißer zu identifizieren und die Notwendigkeit von Datentransformationen zu bewerten.

### 1. AMT_INCOME_TOTAL (Gesamteinkommen)
* **Beobachtung:** Das Diagramm besteht scheinbar nur aus einem einzigen Balken ganz links bei 0, obwohl die x-Achse bis über $1.2 \times 10^8$ (120 Millionen) reicht.
* **Interpretation:** Dies ist ein klassischer Indikator für **extreme Ausreißer**. Fast alle Antragsteller haben ein normales Einkommen (z. B. 30.000 bis 100.000), aber wenige Einzelpersonen im Datensatz haben unrealistisch hohe Werte (entweder Multimillionäre oder, wahrscheinlicher, Tippfehler bei der Dateneingabe, z. B. jemand hat sein Einkommen in Cent statt Euro angegeben).
* **To-Do für das Modell:** Ausreißerbehandlung oder eine Log-Transformation anwenden, da lineare Modelle sonst massiv verzerrt werden.

### 2. AMT_CREDIT (Kreditsumme)
* **Beobachtung:** Eine typische **rechtsschiefe** Verteilung. Der Großteil der Kredite ballt sich im Bereich zwischen 200.000 und 500.000 (vermutlich in der lokalen Währung).
* **Interpretation:** Die Bank vergibt hauptsächlich kleine bis mittlere Konsumkredite. Es gibt einen langen "Schwanz" (Tail) nach rechts bis zu 4 Millionen für seltene, sehr hohe Kredite (z. B. Immobilien).

### 3. age_years (Alter in Jahren)
* **Beobachtung:** Die Verteilung ist relativ flach und breit gefächert zwischen ca. 21 und 69 Jahren. 
* **Interpretation:** Dies ist eine Plausibilitätsprüfung. Im Originaldatensatz wird das Alter kryptisch in negativen Tagen (`DAYS_BIRTH`) angegeben. Dieser Plot beweist, dass das Feature Engineering (Umrechnung in Jahre) fehlerfrei funktioniert hat. Alle Altersgruppen im erwerbsfähigen Alter sind gut repräsentiert.

### 4. credit_income_ratio (Verhältnis Kredit zu Einkommen)
* **Beobachtung:** Extrem rechtsschief. Die meisten Werte liegen zwischen 1 und 10.
* **Interpretation:** Die meisten Antragsteller nehmen Kredite auf, die dem 1- bis 10-fachen ihres Jahreseinkommens entsprechen. Es gibt jedoch Ausreißer, die Kredite in Höhe des 80-fachen ihres Einkommens beantragen. Solche extremen Ratios sind oft starke Warnsignale für ein Ausfallrisiko.

### 5. Der externe Bonitätsscore: `external_score_mean`

> **Parameter-Definition:** `"external_score_mean": ("Externer Bonitätsscore (0-1)", 0.0, 1.0, 0.01)`

* **Beobachtung:** Eine fast glockenförmige Kurve, die jedoch leicht **linksschief (left-skewed)** ist. Die Werte reichen exakt von 0.0 bis 1.0, wobei die Spitze (der Modus) bei etwa 0.6 liegt.
* **Interpretation:** Dieses Feature ist der Durchschnitt der drei externen Schufa-ähnlichen Scores (`EXT_SOURCE_1`, `2` und `3`). 
  * Der Skalenbereich von **0.0 bis 1.0** zeigt, dass die Daten bereits normalisiert sind (vermutlich bedeutet 1.0 = exzellente Bonität, 0.0 = katastrophale Bonität). 
  * Die Linksschiefe ist logisch und positiv für die Bank: Der Großteil der Antragsteller (die Masse unter der Kurve) hat einen "guten" Score zwischen 0.4 und 0.8. Nur ein kleiner Teil (der flache Ausläufer nach links) hat extrem schlechte Werte nahe 0.0. Dies ist oft das stärkste Feature im gesamten Modell, um Kreditausfälle vorherzusagen.

### 6. INST_LATE_PAYMENT_RATIO (Quote verspäteter Ratenzahlungen)
* **Beobachtung:** Fast alle Datenpunkte kleben an der Null-Linie.
* **Interpretation:** Die absolute Mehrheit der Kunden zahlt historische Raten pünktlich (Verzugsquote = 0.0 oder 0 %). Nur eine kleine Gruppe hat Ratios größer als 0. Das ist ein extrem trennscharfes Feature: Wer hier einen Wert weit über Null hat, fällt statistisch gesehen mit sehr hoher Wahrscheinlichkeit auch beim neuen Kredit aus.

In [4]:
num_features = ["AMT_INCOME_TOTAL", "AMT_CREDIT", "age_years",
                "credit_income_ratio", "external_score_mean", "INST_LATE_PAYMENT_RATIO"]
fig = eda.plot_numeric_distributions(feat, num_features); plt.show()

21:15:04 | INFO    | Abbildung gespeichert: C:\Users\annis\OneDrive - Hochschule Heilbronn\MWI\Sem1\ML\files\reports\figures\eda\numeric_distributions.png
C:\Users\annis\AppData\Local\Temp\ipykernel_22300\1430031709.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig = eda.plot_numeric_distributions(feat, num_features); plt.show()


## 4. Ausfall vs. Nicht-Ausfall (Gruppenvergleich)

**Was:** Boxplots zentraler Features getrennt nach TARGET.  
**Warum:** Erste Hinweise auf prädiktive Merkmale.  
## Interpretation

Die vorliegenden Boxplots vergleichen die Verteilung zentraler Merkmale getrennt nach unserer Zielvariable (`TARGET`). Ziel ist es, die **Trennschärfe (Predictive Power)** der einzelnen Variablen visuell zu bewerten. Ein Merkmal ist dann für das Machine-Learning-Modell  wichtig, wenn sich die Boxen (insbesondere die Mediane) der beiden Gruppen deutlich voneinander unterscheiden.

### 1. external_score_mean (Externer Bonitätsscore)
* **Beobachtung:** Es gibt eine sehr deutliche Verschiebung zwischen den Gruppen. Der Median für Nicht-Ausfälle (0) liegt bei ca. 0.55, während er bei den Ausfällen (1) signifikant niedriger bei ca. 0.40 liegt.
* **Interpretation:** Dieses Feature besitzt eine **gute Trennschärfe**. Ein niedriger externer Bonitätsscore korreliert stark mit einem späteren Kreditausfall. Für das ML-Modell wird dies mit hoher Wahrscheinlichkeit eines der wichtigsten Features (Top Feature Importance) sein.

### 2. credit_income_ratio (Kreditsumme im Verhältnis zum Einkommen)
* **Beobachtung:** Die Boxen und Mediane der beiden Gruppen sind nahezu identisch und überlagern sich fast vollständig.
* **Interpretation:** Entgegen der intuitiven Annahme, dass ein relativ zum Einkommen sehr hoher Kredit automatisch zu mehr Ausfällen führt, zeigt dieses Feature isoliert betrachtet **kaum Trennschärfe**. Beide Gruppen verschulden sich im Schnitt mit dem 3- bis 4-fachen ihres Jahreseinkommens. 

### 3. age_years (Alter in Jahren)
* **Beobachtung:** Der Median der Ausfall-Gruppe (orange) ist sichtbar nach unten verschoben (ca. 39 Jahre) im Vergleich zur Nicht-Ausfall-Gruppe (ca. 43 Jahre). Auch das untere Quartil liegt bei den Ausfällen deutlich tiefer.
* **Interpretation:** **Jüngere Antragsteller haben ein statistisch höheres Ausfallrisiko.** Dies ist ein typisches Muster in der Kreditrisikobewertung, da jüngere Personen oft über weniger finanzielle Rücklagen, eine kürzere Kredithistorie und weniger berufliche Stabilität verfügen.

### 4. annuity_income_ratio (Jährliche Rate im Verhältnis zum Einkommen)
* **Beobachtung:** Es gibt nur einen marginalen Unterschied. Die Ausfall-Gruppe hat einen minimal höheren Median und eine leicht nach oben verschobene Box.
* **Interpretation:** Eine höhere monatliche/jährliche Belastung des Einkommens führt zu einer leichten Tendenz des Kreditausfalls. Da der Unterschied jedoch gering ist, wird dieses Feature das Modell zwar unterstützen, aber nicht dominieren.

### 5. AMT_CREDIT (Kreditsumme)
* **Beobachtung:** Interessanterweise liegt der Median der Kreditsumme bei den Personen, die *ausfallen* (1), leicht **niedriger** als bei den zuverlässigen Zahlern (0).
* **Interpretation:** Das wirkt auf den ersten Blick paradox ("Kleine Kredite fallen häufiger aus?"), ist aber bankenlogisch schlüssig: Die Bank hat bereits interne Risikoprüfungen. Risikoreichere Kunden bekommen von vornherein nur kleinere Summen genehmigt. Große Kredite (z. B. Baufinanzierungen) werden nur an Kunden mit exzellenter Bonität und Sicherheiten vergeben, weshalb sie seltener ausfallen.

### 6. INST_LATE_PAYMENT_RATIO (Quote verspäteter Ratenzahlungen)
* **Beobachtung:** Ein massiver visueller Unterschied! Bei den zuverlässigen Kunden (0) klebt der Median auf der Null-Linie und die Box ist sehr schmal. Bei den Ausfällen (1) springt der Median deutlich nach oben, und die Box zieht sich weit in die Höhe.
* **Interpretation:** **Historisches Verhalten ist der beste Prädiktor für zukünftiges Verhalten.** Kunden, die bei früheren Krediten häufig in Zahlungsverzug gerieten, fallen mit extrem hoher Wahrscheinlichkeit auch beim aktuellen Kredit aus. Neben dem externen Score ist dies das stärkste Feature in dieser Analyse.


**Grenzen:**
bivariat, ohne Wechselwirkungen; rein korrelativ.

In [5]:
cmp_features = ["external_score_mean", "credit_income_ratio", "age_years",
                "annuity_income_ratio", "AMT_CREDIT", "INST_LATE_PAYMENT_RATIO"]
fig = eda.plot_target_comparison(feat, cmp_features); plt.show()

21:15:24 | INFO    | Abbildung gespeichert: C:\Users\annis\OneDrive - Hochschule Heilbronn\MWI\Sem1\ML\files\reports\figures\eda\target_comparison_boxplots.png
C:\Users\annis\AppData\Local\Temp\ipykernel_22300\1689097759.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig = eda.plot_target_comparison(feat, cmp_features); plt.show()


## 5. Korrelationen

**Was:** Heatmap der 15 am stärksten mit TARGET (linear) korrelierten Features +
Rangliste.  
**Warum:** Identifiziert lineare Zusammenhänge und Multikollinearität.  
## Interpretation der Korrelations-Heatmap (Pearson)

Die Heatmap visualisiert die linearen Zusammenhänge (Pearson-Korrelation) zwischen den wichtigsten numerischen Merkmalen und der Zielvariable. Dunkelrot steht für eine perfekte positive Korrelation (+1), Dunkelblau für eine perfekte negative Korrelation (-1).

### 1. Beziehungen zur Zielvariable (`TARGET`)
* **Beobachtung:** Die oberste Zeile (bzw. linke Spalte) zeigt die Korrelation aller Features mit dem Kreditausfall (`TARGET`). Es dominieren sehr helle Farben (Werte nahe 0). Die stärksten (negativen) Korrelationen weisen die externen Bonitätsscores (`external_score_mean`, `EXT_SOURCE_2`, etc.) auf (hellblau).
* **Interpretation:** Ein höherer externer Score korreliert mit einem geringeren Ausfallrisiko. Dass keine einzelne Variable eine extrem starke Korrelation (z. B. > 0.5) mit dem `TARGET` aufweist, ist typisch für reale Kreditrisikodaten. Es zeigt, dass einfache lineare Modelle (wie die logistische Regression) hier an ihre Grenzen stoßen werden. Es braucht komplexe, nicht-lineare Algorithmen (wie Random Forest oder XGBoost), die das Zusammenspiel *vieler* schwacher Merkmale kombinieren, um den Ausfall präzise vorherzusagen.

### 2. Identifikation von Multikollinearität (Redundante Features)
Die Heatmap hilft, um Merkmale zu finden, die exakt dieselbe Information enthalten (Multikollinearität).

* **Die dunkelblauen Blöcke (-1.0 Korrelation):**
  * `DAYS_BIRTH` und `age_years` weisen eine perfekte negative Korrelation auf.
  * `DAYS_EMPLOYED` und `employment_years` weisen ebenfalls eine perfekte negative Korrelation auf.
  * *Handlungsableitung:* Das ist ein mathematisch logisches Resultat unseres Feature Engineerings (z. B. Alter in Jahren = `-DAYS_BIRTH / 365`). Für das finale Modelltraining müssen wir die unleserlichen Original-Features (`DAYS_BIRTH`, `DAYS_EMPLOYED`) **zwingend löschen (droppen)**, da sie dem Modell keinen neuen Informationswert mehr liefern.

* **Die dunkelroten Blöcke (hohe positive Korrelation):**
  * Wir sehen starke Cluster zwischen den originalen `EXT_SOURCE`-Variablen und unseren aggregierten Merkmalen (`external_score_mean`, `max`, `min`). 
  * Ebenfalls stark korreliert sind `BUREAU_MEAN_DAYS_CREDIT` und `BUREAU_MIN_DAYS_CREDIT`.
  * *Handlungsableitung:* Lineare Modelle reagieren sehr empfindlich auf solche Abhängigkeiten. Bei baumbasierten Modellen (wie XGBoost) ist das weniger kritisch, dennoch können wir im Rahmen der *Feature Selection* evaluieren, ob das Modell besser performt, wenn wir nur den Durchschnitt (`mean`) behalten und die Einzelwerte entfernen.

**Grenzen:** Pearson erfasst nur lineare Zusammenhänge; Korrelation ≠ Kausalität.

In [6]:
fig, corr = eda.plot_correlation_heatmap(feat); plt.show()
tbl = eda.target_correlation_table(feat, top=15)
utils.save_table(tbl, "target_correlations", subdir="eda")
tbl

21:19:47 | INFO    | Abbildung gespeichert: C:\Users\annis\OneDrive - Hochschule Heilbronn\MWI\Sem1\ML\files\reports\figures\eda\correlation_heatmap.png
C:\Users\annis\AppData\Local\Temp\ipykernel_22300\1780713247.py:1: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig, corr = eda.plot_correlation_heatmap(feat); plt.show()
21:20:00 | INFO    | Tabelle gespeichert: C:\Users\annis\OneDrive - Hochschule Heilbronn\MWI\Sem1\ML\files\reports\tables\eda\target_correlations.csv


,feature,abs_corr,corr
0,external_score_mean,0.222052,-0.222052
1,external_score_max,0.196876,-0.196876
2,external_score_min,0.185266,-0.185266
3,EXT_SOURCE_3,0.178919,-0.178919
4,EXT_SOURCE_2,0.160472,-0.160472
5,EXT_SOURCE_1,0.155317,-0.155317
6,BUREAU_MEAN_DAYS_CREDIT,0.089729,0.089729
7,DAYS_BIRTH,0.078239,0.078239
8,age_years,0.078234,-0.078234
9,BUREAU_MIN_DAYS_CREDIT,0.075248,0.075248


## 6. Erste Hypothesen (datengestützt, vorläufig)

Aus der EDA lassen sich u. a. folgende **Hypothesen** ableiten – als
Eingangspunkt für die Modellierung, **nicht** als gesicherte Befunde:

1. **Externe Scores** sind die stärksten (negativen) Korrelate des Ausfalls –
   höhere Bonitätsscores gehen mit geringerem Risiko einher.
2. Eine hohe **Kredit-Einkommens-Relation** (`credit_income_ratio`) erhöht tendenziell
   das Risiko (Schuldentragfähigkeit).
3. **Zahlungsverhalten** (`INST_LATE_PAYMENT_RATIO`) könnte zusätzliche, von den Scores
   unabhängige Information liefern.
